In [1]:
import sys
import os

import pandas as pd
import scanpy as sc
import numpy as np
import warnings

import torch
import torch.nn.functional as F
import torch.nn as nn

from torch.utils.data import DataLoader, TensorDataset
from torch.nn import DataParallel
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import anndata
import seaborn as sns
import matplotlib.font_manager
from matplotlib import rcParams

font_list = []
fpaths = matplotlib.font_manager.findSystemFonts()
for i in fpaths:
    try:
        f = matplotlib.font_manager.get_font(i)
        font_list.append(f.family_name)
    except RuntimeError:
        pass

font_list = set(font_list)
plot_font = 'Helvetica' if 'Helvetica' in font_list else 'FreeSans'
rcParams['font.family'] = plot_font
rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'figure.figsize': (3,3)})
rcParams.update({'savefig.dpi': 500})
warnings.filterwarnings('ignore')

%load_ext autoreload
%autoreload 2
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [2]:
test_data = sc.read_h5ad('gears_test_data.h5ad')

In [3]:
print("test data shape:", test_data.shape)
print("Group in test data:", test_data.obs["Group"].unique())

# shuffle indices
total_indices = np.arange(test_data.shape[0])
np.random.shuffle(total_indices)

# 50% train, 50% test
size = int(0.5 * test_data.shape[0])

train_indices = total_indices[:size]
test_indices = total_indices[size:]

sub_train_set = test_data[train_indices, :].copy()
test_set = test_data[test_indices, :].copy()

print("Train:", sub_train_set.shape)
print("Test:", test_set.shape)


test data shape: (303, 33694)
Group in test data: [3]
Train: (151, 33694)
Test: (152, 33694)


In [4]:
train_data = sc.read_h5ad('gears_train_data.h5ad')
combined = anndata.concat([train_data, sub_train_set])
print(combined.obs['Group'].unique())
combined.write_h5ad(f'new_train_data.h5ad')

[0 2 1 3]


In [5]:
print(train_data)
print(combined)

AnnData object with n_obs × n_vars = 7890 × 33694
    obs: 'Group'
    var: 'gene_ids'
    uns: 'log1p'
AnnData object with n_obs × n_vars = 8041 × 33694
    obs: 'Group'


In [6]:

test_set.write_h5ad(
    f"test_data.h5ad"
)